# 01_c_add_features

> Purpose: extend the preprocessed dataset with feature engineering that helps segmentation and modeling.

This notebook loads `data/processed/preprocessed_data.csv`, creates derived features for customer profile and channel behavior, then saves an enriched dataset to `data/features_added/preprocessed_data_features_added.csv`.

## Step 1) Load preprocessed input data

> Reads `data/processed/preprocessed_data.csv`, validates file availability, and displays a quick preview for sanity checking.

In [10]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)

PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'

input_path = DATA_DIR / 'processed' / 'preprocessed_data.csv'
if not input_path.exists():
    raise FileNotFoundError(f'Input file not found: {input_path}')

df = pd.read_csv(input_path)
print('Loaded:', input_path)
print('Shape:', df.shape)
df.head()

Loaded: d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_data_driven_marketing\starbucks_reward_data_marketing\data\processed\preprocessed_data.csv
Shape: (50637, 50)


,person,offer_id,time_completed,time_received,time_viewed,was_viewed,completed_after_view,within_offer_window,offer_success,received_not_viewed,age,income,days_since_registration,age_was_118,income_missing_before_impute,gender_missing_before_fill,age_missing_before_impute,reward,difficulty,channel_email,channel_mobile,channel_social,channel_web,channel_count,reward_to_difficulty,reward_per_day,offer_type_bogo,offer_type_discount,offer_type_informational,transaction_count,total_spent,avg_transaction_value,duration,time_completed_was_imputed,time_viewed_was_imputed,age_group,income_group,membership_duration_months,has_web_channel,has_email_channel,has_mobile_channel,has_social_channel,spend_per_transaction,spend_per_membership_day,difficulty_per_day,net_value_score,gender_F,gender_M,gender_O,gender_U
0,0009655768c64bdeb2e877511632db8f,2906b810c7d4411798c6938adc9daaa5,576.0,576.0,360.0,0,False,True,0,1,33,72000.0,461,0,0,0,0,2,10,1,1,0,1,3,0.20,0.285714,0,1,0,8.0,127.60,15.950000,7,0,1,26-35,middle,15.14,1,1,1,0,15.950000,0.276790,1.428571,1.0,False,True,False,False
1,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,414.0,408.0,456.0,1,False,True,0,0,33,72000.0,461,0,0,0,0,5,5,1,1,1,1,4,1.00,1.000000,1,0,0,8.0,127.60,15.950000,5,0,0,26-35,middle,15.14,1,1,1,1,15.950000,0.276790,1.000000,4.5,False,True,False,False
2,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,528.0,504.0,540.0,1,False,True,0,0,33,72000.0,461,0,0,0,0,2,10,1,1,1,1,4,0.20,0.200000,0,1,0,8.0,127.60,15.950000,10,0,0,26-35,middle,15.14,1,1,1,1,15.950000,0.276790,1.000000,1.0,False,True,False,False
3,00116118485d4dfda04fdbaba9a87b5c,f19421c1d4aa40978ebb69ca19b0e20d,420.0,168.0,216.0,1,False,False,0,0,55,64000.0,92,1,1,1,1,5,5,1,1,1,1,4,1.00,1.000000,1,0,0,3.0,4.09,1.363333,5,1,0,46-55,middle,3.02,1,1,1,1,1.363333,0.044457,1.000000,4.5,False,False,False,True
4,0011e0d4e6b944f998e987f904e8c1e5,0b1e1539f2cc45b7b9fa7c272da2e1d7,576.0,408.0,432.0,1,True,True,1,0,40,57000.0,198,0,0,0,0,5,20,1,0,0,1,2,0.25,0.500000,0,1,0,5.0,79.46,15.892000,10,0,0,36-45,lower_middle,6.50,1,1,0,0,15.892000,0.401313,2.000000,3.0,False,False,True,False


## Step 2) Engineer additional features

> Adds `membership_duration_months`, creates `age_group` and `income_group`, and builds channel availability flags from existing channel columns (`web`, `email`, `mobile`, `social`).

In [11]:
# Membership duration in months
if 'days_since_registration' not in df.columns:
    raise KeyError("Expected column 'days_since_registration' in input data")

df['membership_duration_months'] = (df['days_since_registration'] / 30.44).round(2)

# Age groups
if 'age' not in df.columns:
    raise KeyError("Expected column 'age' in input data")

age_bins = [0, 25, 35, 45, 55, 65, np.inf]
age_labels = ['18-25', '26-35', '36-45', '46-55', '56-65', '66+']
df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels, include_lowest=True)

# Income groups
if 'income' not in df.columns:
    raise KeyError("Expected column 'income' in input data")

income_bins = [0, 40000, 60000, 80000, 100000, np.inf]
income_labels = ['low', 'lower_middle', 'middle', 'upper_middle', 'high']
df['income_group'] = pd.cut(df['income'], bins=income_bins, labels=income_labels, include_lowest=True)

# Channel flags from either legacy columns (web/email/mobile/social) or new schema (channel_*)
channel_candidates = {
    'web': ['web', 'channel_web'],
    'email': ['email', 'channel_email'],
    'mobile': ['mobile', 'channel_mobile'],
    'social': ['social', 'channel_social']
}

resolved_channels = {}
for logical_name, candidates in channel_candidates.items():
    found_col = next((c for c in candidates if c in df.columns), None)
    if found_col is None:
        raise KeyError(f"Expected one of {candidates} in input data for '{logical_name}' channel")
    resolved_channels[logical_name] = found_col

df['has_web_channel'] = (df[resolved_channels['web']] > 0).astype(int)
df['has_email_channel'] = (df[resolved_channels['email']] > 0).astype(int)
df['has_mobile_channel'] = (df[resolved_channels['mobile']] > 0).astype(int)
df['has_social_channel'] = (df[resolved_channels['social']] > 0).astype(int)

print('Feature engineering completed.')
print('Resolved channel columns:', resolved_channels)
print('New columns added:', [
    'membership_duration_months', 'age_group', 'income_group',
    'has_web_channel', 'has_email_channel', 'has_mobile_channel', 'has_social_channel'
])

Feature engineering completed.
Resolved channel columns: {'web': 'channel_web', 'email': 'channel_email', 'mobile': 'channel_mobile', 'social': 'channel_social'}
New columns added: ['membership_duration_months', 'age_group', 'income_group', 'has_web_channel', 'has_email_channel', 'has_mobile_channel', 'has_social_channel']


## Step 3) Export enriched dataset

> Converts group features to CSV-safe strings and saves the final dataset to `data/features_added/preprocessed_data_features_added.csv`.

In [12]:
output_dir = DATA_DIR / 'features_added'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'preprocessed_data_features_added.csv'

# Convert category columns to string labels for CSV compatibility
df['age_group'] = df['age_group'].astype(str)
df['income_group'] = df['income_group'].astype(str)

df.to_csv(output_path, index=False)

print('Saved:', output_path)
print('Final shape:', df.shape)
print('Preview of added feature columns:')
df[[
    'membership_duration_months', 'age_group', 'income_group',
    'has_web_channel', 'has_email_channel', 'has_mobile_channel', 'has_social_channel'
]].head()

Saved: d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_data_driven_marketing\starbucks_reward_data_marketing\data\features_added\preprocessed_data_features_added.csv
Final shape: (50637, 50)
Preview of added feature columns:


,membership_duration_months,age_group,income_group,has_web_channel,has_email_channel,has_mobile_channel,has_social_channel
0,15.14,26-35,middle,1,1,1,0
1,15.14,26-35,middle,1,1,1,1
2,15.14,26-35,middle,1,1,1,1
3,3.02,46-55,middle,1,1,1,1
4,6.50,36-45,lower_middle,1,1,0,0
